In [294]:
import numpy as np

In [295]:
def relu_back(x):
    return x>0

In [296]:


class Matrix:
    def __init__(self, data, children = None):
        self.data = np.array(data, dtype=float)
        self.grad = np.zeros_like(self.data, dtype=float)
        self.children = children

    def matmul(self,other):
        out = Matrix(np.array(self.data@other.data), children = [self, other, "matmul"])
        return out

    def matsum(self,other):
        out = Matrix(np.array(self.data+other.data), children = [self,other,"matsum"])
        return out

    def matrelu(self):
        def relu(x):
            return np.maximum(0,x)
        out = Matrix(np.array(relu(self.data)), children = [self, "matrelu"])
        return out

    def matsub(self,other):
        out = Matrix(np.array(self.data-other.data), children = [self,other,  "matsub"])
        return out

    def matdiv(self,other):
        out = Matrix(np.array(self.data/other.data), children = [self,other, "matdiv"])
        return out

    def matexp(self):
        out = Matrix(np.array(np.exp(self.data)), children = [self,"matexp"])
        return out

    def matlog(self):
        out = Matrix(np.array(np.log(self.data)),children = [self, "matlog"])
        return out

    def shape(self):
            return np.array(self.data).shape

    def softmax_cross_entropy(self, label): #block for softmax and cross entropy instead of just primary operations
        def softmax(x):
            x = np.exp(x)
            return x/np.sum(x,axis=1, keepdims=True)
        probabilities = softmax(self.data)
        n = self.data.shape[0]
        loss = -(np.sum(label.data*np.log(probabilities)))/n

        out = Matrix(np.array(loss), children = [self,label, "softmax_cross_entropy"])
        return out, probabilities
        
    def backprop(self, probabilities):
        topo = []  #topo algorithm is just copied
        visited = set()
        def build_topological_order(node):
            if node not in visited:
                visited.add(node)
                if node.children:
                    for child in node.children[:len(node.children)-1]:
                        build_topological_order(child)
        
                topo.append(node)
        build_topological_order(self)
        for node in topo:
            node.grad = np.zeros_like(node.data, dtype=float)

        self.grad = np.ones_like(self.data, dtype=float)
        for mat in reversed(topo):
                    if not mat.children:
                        continue
                    op = mat.children[-1]
                    if op!="matrelu" and op!="matexp" and op!="matlog":
                        left_child = mat.children[0]
                        right_child = mat.children[1]
                        if op == "matmul":
                            left_child.grad +=  mat.grad @ right_child.data.T
                            right_child.grad += left_child.data.T @ mat.grad
                        elif op == "matsum":
                            left_child.grad += mat.grad 
                            right_child.grad += mat.grad.sum(0) #I assume that the right child is the bias here and will always be a vector
                        elif op == "matsub":
                            left_child.grad += mat.grad
                            right_child.grad -= mat.grad
                        elif op == "matdiv":
                            left_child.grad += mat.grad * (right_child.data**-1)
                            right_child.grad += mat.grad * (-left_child.data / right_child.data**2)
                        elif op == "softmax_cross_entropy":
                            probs = probabilities
                            n = left_child.data.shape[0]
                            left_child.grad += mat.grad * (probs - right_child.data) / n

                    else:
                        child = mat.children[0]
                        if op == "matrelu":
                            child.grad += mat.grad * relu_back(child.data)
                        elif op =="matexp":
                            child.grad += mat.grad * mat.data
                        elif op =="matlog":
                            child.grad += mat.grad * (1/child.data)
                        
    def __repr__(self):
        return f"data = {self.data}, grad = {self.grad}"

